# GBM vs Random Forest using XGBoost in Python

This notebook demonstrates the key differences between **Gradient Boosting (GBM)** and **Random Forest** implementations using XGBoost.

## Key Differences:
- **Boosting (GBM)**: `num_parallel_tree=1`, `n_estimators=many` - Trees added sequentially
- **Random Forest**: `num_parallel_tree=many`, `n_estimators=1` - Trees built independently in parallel

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import time
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    roc_auc_score, 
    classification_report,
    mean_squared_error,
    r2_score
)
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"XGBoost version: {XGBClassifier().get_xgb_params()['base_score']}")

## 2. Load Data from Online Source

In [ ]:
# Load data from online source
file_path = 'https://raw.githubusercontent.com/axspinnacle/cfs25/main/data/'
file_name = 'data5.ftr'

print("Loading data from online source...")
try:
    df_ini = pd.read_feather(file_path + file_name)
    print(f"✓ Data loaded successfully!")
    print(f"  Shape: {df_ini.shape}")
    print(f"  Columns: {len(df_ini.columns)}")
except Exception as e:
    print(f"✗ Error loading data: {e}")
    print("Creating sample data for demonstration...")
    np.random.seed(42)
    n_samples = 5000
    df_ini = pd.DataFrame({
        'feature1': np.random.normal(0, 1, n_samples),
        'feature2': np.random.normal(2, 0.5, n_samples),
        'feature3': np.random.randint(0, 5, n_samples),
        'feature4': np.random.exponential(1, n_samples),
        'feature5': np.random.uniform(0, 10, n_samples),
        'cc_col': np.random.randint(0, 2, n_samples)  # Binary target
    })
    print(f"  Sample data created with shape: {df_ini.shape}")

# Create working copy
df = df_ini.copy()
print("\nFirst few rows:")
df.head()

## 3. Data Preprocessing

In [ ]:
# Remove unnecessary columns (if they exist)
cols_to_remove = ['ep_bi', 'ep_col', 'ee_bi', 'ee_col', 'incloss_bi', 'incloss_col', 'cc_bi', 'zip', 'pol_id', 'vin_id', 'Date']
existing_cols_to_remove = [col for col in cols_to_remove if col in df.columns]

if existing_cols_to_remove:
    df = df.drop(columns=existing_cols_to_remove)
    print(f"Removed columns: {existing_cols_to_remove}")

# Define target column
target_column = "cc_col"
print(f"\nUsing '{target_column}' as target variable")

# Separate features and target
X = df.drop(columns=[target_column])
y = df[target_column]

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nTarget distribution (%):")
print(y.value_counts(normalize=True) * 100)

In [ ]:
# Handle categorical variables
categorical_columns = X.select_dtypes(include=['object', 'category']).columns
if len(categorical_columns) > 0:
    print(f"Encoding categorical columns: {list(categorical_columns)}")
    le = LabelEncoder()
    for col in categorical_columns:
        X[col] = le.fit_transform(X[col].astype(str))
    print("✓ Categorical variables encoded")
else:
    print("✓ No categorical variables found")

# Handle missing values
if X.isnull().sum().sum() > 0:
    print(f"\nFilling {X.isnull().sum().sum()} missing values...")
    X = X.fillna(X.median())
    print("✓ Missing values handled")
else:
    print("\n✓ No missing values found")

print("\n✓ Data preprocessing complete!")

## 4. Train-Test Split

Using consistent random_state=42 for fair comparison

In [ ]:
# Split data - 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining target distribution:")
print(y_train.value_counts())
print(f"\nTest target distribution:")
print(y_test.value_counts())

## 5. Model Configurations

### Gradient Boosting (GBM) Model
- **Sequential tree building**: Each tree corrects errors of previous trees
- `n_estimators=200` (many boosting rounds)
- `num_parallel_tree=1` (one tree per round)
- `learning_rate=0.05` (slower learning)

In [ ]:
# Determine if binary or multiclass
n_classes = len(y.unique())
is_binary = n_classes == 2

print(f"Problem type: {'Binary' if is_binary else 'Multiclass'} Classification")
print(f"Number of classes: {n_classes}")

# GBM Model Configuration
print("\n" + "="*60)
print("GRADIENT BOOSTING MODEL (GBM)")
print("="*60)

boost_model = XGBClassifier(
    n_estimators=200,           # Many boosting rounds
    num_parallel_tree=1,        # One tree per round (sequential)
    max_depth=6,                # Tree depth
    learning_rate=0.05,         # Slower learning rate
    subsample=0.8,              # Row sampling
    colsample_bytree=0.8,       # Column sampling per tree
    random_state=42,
    eval_metric='logloss' if is_binary else 'mlogloss',
    tree_method='hist'
)

print(f"Configuration:")
print(f"  - n_estimators: 200 (sequential rounds)")
print(f"  - num_parallel_tree: 1")
print(f"  - learning_rate: 0.05")
print(f"  - max_depth: 6")
print(f"  - Total trees: 200 × 1 = 200 trees")
print(f"\nTraining GBM model...")

start_time = time.time()
boost_model.fit(X_train, y_train, verbose=False)
boost_train_time = time.time() - start_time

print(f"✓ Training completed in {boost_train_time:.2f} seconds")

### Random Forest Model
- **Parallel tree building**: All trees built independently
- `n_estimators=1` (single boosting round)
- `num_parallel_tree=200` (200 trees built in parallel)
- `learning_rate=1` (no shrinkage)

In [ ]:
print("="*60)
print("RANDOM FOREST MODEL (using XGBoost)")
print("="*60)

rf_model = XGBClassifier(
    n_estimators=1,             # Single boosting iteration
    num_parallel_tree=200,      # 200 trees in parallel
    max_depth=6,                # Same tree depth
    learning_rate=1,            # No shrinkage
    subsample=0.8,              # Row sampling (bagging)
    colsample_bynode=0.8,       # Feature sampling per split (like RF)
    random_state=42,
    eval_metric='logloss' if is_binary else 'mlogloss',
    tree_method='hist'
)

print(f"Configuration:")
print(f"  - n_estimators: 1 (single round)")
print(f"  - num_parallel_tree: 200")
print(f"  - learning_rate: 1.0")
print(f"  - max_depth: 6")
print(f"  - Total trees: 1 × 200 = 200 trees")
print(f"\nTraining Random Forest model...")

start_time = time.time()
rf_model.fit(X_train, y_train, verbose=False)
rf_train_time = time.time() - start_time

print(f"✓ Training completed in {rf_train_time:.2f} seconds")

## 6. Model Predictions and Metrics

We'll evaluate both models on:
- Train and Test sets
- Calculate AUC (for classification)
- Measure inference time

In [ ]:
# GBM Predictions
print("Making predictions with GBM model...")
start_time = time.time()
boost_train_pred = boost_model.predict(X_train)
boost_test_pred = boost_model.predict(X_test)
boost_inference_time = time.time() - start_time

# Get probability predictions for AUC
boost_train_proba = boost_model.predict_proba(X_train)
boost_test_proba = boost_model.predict_proba(X_test)

print(f"✓ GBM predictions completed in {boost_inference_time:.4f} seconds")

# Random Forest Predictions
print("\nMaking predictions with Random Forest model...")
start_time = time.time()
rf_train_pred = rf_model.predict(X_train)
rf_test_pred = rf_model.predict(X_test)
rf_inference_time = time.time() - start_time

# Get probability predictions for AUC
rf_train_proba = rf_model.predict_proba(X_train)
rf_test_proba = rf_model.predict_proba(X_test)

print(f"✓ RF predictions completed in {rf_inference_time:.4f} seconds")

In [ ]:
# Calculate metrics
if is_binary:
    # Binary classification - use column 1 for positive class
    boost_train_auc = roc_auc_score(y_train, boost_train_proba[:, 1])
    boost_test_auc = roc_auc_score(y_test, boost_test_proba[:, 1])
    rf_train_auc = roc_auc_score(y_train, rf_train_proba[:, 1])
    rf_test_auc = roc_auc_score(y_test, rf_test_proba[:, 1])
    metric_name = "AUC"
else:
    # Multiclass - use ovr (one-vs-rest)
    boost_train_auc = roc_auc_score(y_train, boost_train_proba, multi_class='ovr')
    boost_test_auc = roc_auc_score(y_test, boost_test_proba, multi_class='ovr')
    rf_train_auc = roc_auc_score(y_train, rf_train_proba, multi_class='ovr')
    rf_test_auc = roc_auc_score(y_test, rf_test_proba, multi_class='ovr')
    metric_name = "AUC (OVR)"

# Calculate accuracies
boost_train_acc = accuracy_score(y_train, boost_train_pred)
boost_test_acc = accuracy_score(y_test, boost_test_pred)
rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

# Calculate generalization gaps
boost_gap = boost_train_auc - boost_test_auc
rf_gap = rf_train_auc - rf_test_auc

print("✓ All metrics calculated successfully!")

## 7. Comprehensive Comparison

### Generalization Gap
The **generalization gap** (Train Metric - Test Metric) is a key indicator of model capacity:
- **Larger gap**: Higher capacity model, potential overfitting
- **Smaller gap**: Better generalization to unseen data

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame({
    'Metric': [
        f'Train {metric_name}',
        f'Test {metric_name}',
        'Generalization Gap',
        'Train Accuracy',
        'Test Accuracy',
        'Total Trees',
        'Max Depth',
        'Training Time (s)',
        'Inference Time (s)'
    ],
    'Gradient Boosting': [
        f"{boost_train_auc:.4f}",
        f"{boost_test_auc:.4f}",
        f"{boost_gap:.4f}",
        f"{boost_train_acc:.4f}",
        f"{boost_test_acc:.4f}",
        "200",
        "6",
        f"{boost_train_time:.4f}",
        f"{boost_inference_time:.4f}"
    ],
    'Random Forest': [
        f"{rf_train_auc:.4f}",
        f"{rf_test_auc:.4f}",
        f"{rf_gap:.4f}",
        f"{rf_train_acc:.4f}",
        f"{rf_test_acc:.4f}",
        "200",
        "6",
        f"{rf_train_time:.4f}",
        f"{rf_inference_time:.4f}"
    ]
})

print("="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

In [ ]:
# Interpretation
print("\n📊 INTERPRETATION:")
print("\n1. GENERALIZATION GAP:")
print(f"   - GBM Gap: {boost_gap:.4f}")
print(f"   - RF Gap: {rf_gap:.4f}")
if boost_gap > rf_gap:
    print("   → GBM shows larger gap, indicating higher model capacity but potentially more overfitting")
else:
    print("   → RF shows larger gap, indicating higher model capacity but potentially more overfitting")

print("\n2. TEST PERFORMANCE:")
print(f"   - GBM Test {metric_name}: {boost_test_auc:.4f}")
print(f"   - RF Test {metric_name}: {rf_test_auc:.4f}")
if boost_test_auc > rf_test_auc:
    print(f"   → GBM performs better on test set by {(boost_test_auc - rf_test_auc):.4f}")
else:
    print(f"   → RF performs better on test set by {(rf_test_auc - boost_test_auc):.4f}")

print("\n3. COMPUTATIONAL EFFICIENCY:")
print(f"   - GBM Training: {boost_train_time:.4f}s | Inference: {boost_inference_time:.4f}s")
print(f"   - RF Training: {rf_train_time:.4f}s | Inference: {rf_inference_time:.4f}s")
if rf_train_time < boost_train_time:
    print(f"   → RF trains {boost_train_time/rf_train_time:.2f}x faster (parallel tree building)")
else:
    print(f"   → GBM trains {rf_train_time/boost_train_time:.2f}x faster")

## 8. Visualizations

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('GBM vs Random Forest Comparison', fontsize=16, fontweight='bold')

# 1. AUC Comparison
ax1 = axes[0, 0]
models = ['GBM', 'RF']
train_scores = [boost_train_auc, rf_train_auc]
test_scores = [boost_test_auc, rf_test_auc]
x = np.arange(len(models))
width = 0.35

ax1.bar(x - width/2, train_scores, width, label='Train', alpha=0.8, color='#2ecc71')
ax1.bar(x + width/2, test_scores, width, label='Test', alpha=0.8, color='#3498db')
ax1.set_ylabel(f'{metric_name} Score', fontweight='bold')
ax1.set_title(f'{metric_name} Score Comparison', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(models)
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(train_scores):
    ax1.text(i - width/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
for i, v in enumerate(test_scores):
    ax1.text(i + width/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

# 2. Generalization Gap
ax2 = axes[0, 1]
gaps = [boost_gap, rf_gap]
colors = ['#e74c3c' if g > 0.05 else '#f39c12' if g > 0.02 else '#2ecc71' for g in gaps]
bars = ax2.bar(models, gaps, color=colors, alpha=0.8)
ax2.set_ylabel('Generalization Gap', fontweight='bold')
ax2.set_title('Generalization Gap (Train - Test)', fontweight='bold')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for i, (bar, gap) in enumerate(zip(bars, gaps)):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.002,
             f'{gap:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 3. Training Time
ax3 = axes[1, 0]
times = [boost_train_time, rf_train_time]
bars = ax3.bar(models, times, color=['#9b59b6', '#1abc9c'], alpha=0.8)
ax3.set_ylabel('Time (seconds)', fontweight='bold')
ax3.set_title('Training Time Comparison', fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

# Add value labels
for bar, time in zip(bars, times):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
             f'{time:.3f}s', ha='center', va='bottom', fontsize=10)

# 4. Accuracy Comparison
ax4 = axes[1, 1]
train_acc = [boost_train_acc, rf_train_acc]
test_acc = [boost_test_acc, rf_test_acc]

ax4.bar(x - width/2, train_acc, width, label='Train', alpha=0.8, color='#e67e22')
ax4.bar(x + width/2, test_acc, width, label='Test', alpha=0.8, color='#16a085')
ax4.set_ylabel('Accuracy', fontweight='bold')
ax4.set_title('Accuracy Comparison', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(models)
ax4.legend()
ax4.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(train_acc):
    ax4.text(i - width/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
for i, v in enumerate(test_acc):
    ax4.text(i + width/2, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 9. Detailed Classification Reports

In [ ]:
print("="*80)
print("GRADIENT BOOSTING - Classification Report")
print("="*80)
print(classification_report(y_test, boost_test_pred))

print("\n" + "="*80)
print("RANDOM FOREST - Classification Report")
print("="*80)
print(classification_report(y_test, rf_test_pred))

## 10. Key Takeaways

### Boosting vs Bagging Comparison:

**Gradient Boosting (GBM):**
- ✅ Often achieves higher test performance
- ✅ Good for squeezing out maximum predictive power
- ⚠️ Higher risk of overfitting (larger generalization gap)
- ⚠️ Sequential training (can be slower)
- ⚠️ More sensitive to hyperparameters

**Random Forest:**
- ✅ Better generalization (smaller gap)
- ✅ More robust, less prone to overfitting
- ✅ Parallel training (can be faster)
- ✅ Less sensitive to hyperparameters
- ⚠️ May have slightly lower test performance

### When to Use Each:

**Use Boosting when:**
- You need maximum predictive performance
- You have time to tune hyperparameters
- You have enough data to prevent overfitting

**Use Random Forest when:**
- You want robust, reliable predictions
- You have limited time for hyperparameter tuning
- You're concerned about overfitting
- You need faster training with parallel processing

## 11. Alternative: Compare with sklearn's RandomForest

For a more traditional comparison, you might also want to compare XGBoost's boosting with scikit-learn's RandomForestClassifier:

In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("Training sklearn's RandomForestClassifier for comparison...")

sklearn_rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    max_features=0.8,
    max_samples=0.8,
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
sklearn_rf.fit(X_train, y_train)
sklearn_rf_train_time = time.time() - start_time

# Predictions
sklearn_rf_test_pred = sklearn_rf.predict(X_test)
sklearn_rf_test_proba = sklearn_rf.predict_proba(X_test)

# Metrics
if is_binary:
    sklearn_rf_test_auc = roc_auc_score(y_test, sklearn_rf_test_proba[:, 1])
else:
    sklearn_rf_test_auc = roc_auc_score(y_test, sklearn_rf_test_proba, multi_class='ovr')

sklearn_rf_test_acc = accuracy_score(y_test, sklearn_rf_test_pred)

print(f"\n✓ sklearn RF Training Time: {sklearn_rf_train_time:.4f}s")
print(f"✓ sklearn RF Test {metric_name}: {sklearn_rf_test_auc:.4f}")
print(f"✓ sklearn RF Test Accuracy: {sklearn_rf_test_acc:.4f}")

print("\n" + "="*80)
print("THREE-WAY COMPARISON")
print("="*80)
print(f"{'Model':<25} {'Test ' + metric_name:<15} {'Test Accuracy':<15} {'Train Time (s)'}")
print("-"*80)
print(f"{'XGBoost Boosting':<25} {boost_test_auc:<15.4f} {boost_test_acc:<15.4f} {boost_train_time:.4f}")
print(f"{'XGBoost RF Mode':<25} {rf_test_auc:<15.4f} {rf_test_acc:<15.4f} {rf_train_time:.4f}")
print(f"{'sklearn RandomForest':<25} {sklearn_rf_test_auc:<15.4f} {sklearn_rf_test_acc:<15.4f} {sklearn_rf_train_time:.4f}")
print("="*80)

## 12. Summary and Conclusions

In [ ]:
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

print("\n🔬 EXPERIMENTAL SETUP:")
print(f"  - Dataset size: {len(df)} samples")
print(f"  - Features: {X.shape[1]}")
print(f"  - Train/Test split: 80/20")
print(f"  - Problem type: {n_classes}-class classification")

print("\n🎯 MODEL CONFIGURATIONS:")
print("  Both models used:")
print("    - 200 total trees")
print("    - max_depth = 6")
print("    - subsample = 0.8")
print("    - colsample = 0.8")
print("    - random_state = 42")

print("\n📈 PERFORMANCE COMPARISON:")
winner = "GBM" if boost_test_auc > rf_test_auc else "Random Forest"
print(f"  Winner: {winner} (higher test {metric_name})")
print(f"  Better generalization: {'Random Forest' if rf_gap < boost_gap else 'GBM'} (smaller gap)")
print(f"  Faster training: {'Random Forest' if rf_train_time < boost_train_time else 'GBM'}")

print("\n💡 RECOMMENDATION:")
if boost_test_auc > rf_test_auc and boost_gap < 0.05:
    print("  → Use Gradient Boosting: Better performance without excessive overfitting")
elif rf_gap < boost_gap and abs(boost_test_auc - rf_test_auc) < 0.02:
    print("  → Use Random Forest: Similar performance with better generalization")
elif boost_test_auc > rf_test_auc:
    print("  → Use Gradient Boosting: Superior test performance")
else:
    print("  → Use Random Forest: Better overall balance")

print("\n" + "="*80)
print("✅ Analysis Complete!")
print("="*80)